# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Logistic Regression** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [12]:
from SparkUtils import SparkUtils

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:

MASTER_URL = "spark://spark-master:7077"
APP_NAME = "Lab 10: Logistic Regression"

spark = SparkUtils(MASTER_URL, APP_NAME)._spark

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 01:12:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Collect Data

In [3]:
# Create a small dataset as a list of tuples
# Format: (label, feature_x1, feature_x2)
data = [
    (1.0, 2.0, 3.0),
    (0.0, 1.0, 2.5),
    (1.0, 3.0, 5.0),
    (0.0, 0.5, 1.0),
    (1.0, 4.0, 6.0)
]

# Define schema for the DataFrame
schema = SparkUtils.generate_schema(
    [
        ("label", "float"), 
        ("feature_x1", "float"),
        ("feature_x2", "float")
    ]
)

# Convert list to a DataFrame
df = spark.createDataFrame(data, schema)

### Assemble the features into a single vector column

In [5]:
assembler = VectorAssembler(inputCols=["feature_x1", "feature_x2"], outputCol="features")

data_with_features = assembler.transform(df).select("label", "features")

data_with_features.printSchema()                                   

root
 |-- label: float (nullable = true)
 |-- features: vector (nullable = true)



# Data splitting
#### 80% training data and 20% testing data

In [6]:
train_df, test_df = data_with_features.randomSplit([0.8, 0.2], seed=67)

### Show dataset (for debugging)

In [7]:
print("Original Dataset")
df.show()

# Print train dataset
print("train set")
train_df.show()

Original Dataset


+-----+----------+----------+
|label|feature_x1|feature_x2|
+-----+----------+----------+
|  1.0|       2.0|       3.0|
|  0.0|       1.0|       2.5|
|  1.0|       3.0|       5.0|
|  0.0|       0.5|       1.0|
|  1.0|       4.0|       6.0|
+-----+----------+----------+

train set


+-----+---------+
|label| features|
+-----+---------+
|  0.0|[1.0,2.5]|
|  1.0|[2.0,3.0]|
|  0.0|[0.5,1.0]|
+-----+---------+



# Create ML Model

In [9]:
lr = LogisticRegression(maxIter=10, regParam=0.01)

# Train ML Model

In [10]:
lr_model = lr.fit(train_df)

# Print coefficients
print("Coefficients: " + str(lr_model.coefficients))

# Display model summary
training_summary = lr_model.summary

26/04/16 01:21:42 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Coefficients: [4.392933013678946,1.2369280013452952]


## Predictions

In [11]:
# Use the trained model to make predictions on the test data
predictions = lr_model.transform(test_df)

# Show predictions
predictions.select("features", "prediction", "probability").show()

+---------+----------+--------------------+
| features|prediction|         probability|
+---------+----------+--------------------+
|[3.0,5.0]|       1.0|[8.59029644215258...|
|[4.0,6.0]|       1.0|[3.08338498567314...|
+---------+----------+--------------------+



# Test ML Model

In [13]:
evaluator = MulticlassClassificationEvaluator(labelCol="label",
                            predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, 
                  {evaluator.metricName: "accuracy"})
print(f"Accuracy: {accuracy}")
precision = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedPrecision"})
print(f"Precision: {precision}")
recall = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedRecall"})
print(f"Recall: {recall}")
f1 = evaluator.evaluate(predictions,
                {evaluator.metricName: "f1"})
print(f"F1 Score: {f1}")  

Accuracy: 1.0


Precision: 1.0
Recall: 1.0
F1 Score: 1.0


# Lab 10: Logistic regression to predict heart disease

# Data collection

In [ ]:
# Define schema for the DataFrame
heart_schema = SparkUtils.generate_schema(
    [
        ("male", "int"), 
        ("age", "int"), 
        ("education", "int"), 
        ("currentSmoker", "int"), 
        ("cigsPerDay", "int"), 
        ("BPMeds", "int"), 
        ("prevalentStroke", "int"), 
        ("prevalentHyp", "int"), 
        ("diabetes", "int"), 
        ("totChol", "int"), 
        ("sysBP", "float"), 
        ("diaBP", "float"), 
        ("BMI", "float"), 
        ("heartRate", "int"), 
        ("glucose", "int"), 
        ("TenYearCHD", "int")
    ]
)

# Source: https://www.kaggle.com/datasets/dileep070/heart-disease-prediction-using-logistic-regression?resource=download

heart_df = spark.read \
    .option("header", "true") \
    .schema(heart_schema) \
    .csv("/opt/spark/work-dir/data/ml/logistic_regression/framingham.csv")

heart_df.printSchema()

root
 |-- male: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- education: integer (nullable = true)
 |-- currentSmoker: integer (nullable = true)
 |-- cigsPerDay: integer (nullable = true)
 |-- BPMeds: integer (nullable = true)
 |-- prevalentStroke: integer (nullable = true)
 |-- prevalentHyp: integer (nullable = true)
 |-- diabetes: integer (nullable = true)
 |-- totChol: integer (nullable = true)
 |-- sysBP: float (nullable = true)
 |-- diaBP: float (nullable = true)
 |-- BMI: float (nullable = true)
 |-- heartRate: integer (nullable = true)
 |-- glucose: integer (nullable = true)
 |-- TenYearCHD: integer (nullable = true)



In [15]:
heart_df.show(2)

+----+---+---------+-------------+----------+------+---------------+------------+--------+-------+-----+-----+-----+---------+-------+----------+
|male|age|education|currentSmoker|cigsPerDay|BPMeds|prevalentStroke|prevalentHyp|diabetes|totChol|sysBP|diaBP|  BMI|heartRate|glucose|TenYearCHD|
+----+---+---------+-------------+----------+------+---------------+------------+--------+-------+-----+-----+-----+---------+-------+----------+
|   1| 39|        4|            0|         0|     0|              0|           0|       0|    195|106.0| 70.0|26.97|       80|     77|         0|
|   0| 46|        2|            0|         0|     0|              0|           0|       0|    250|121.0| 81.0|28.73|       95|     76|         0|
+----+---+---------+-------------+----------+------+---------------+------------+--------+-------+-----+-----+-----+---------+-------+----------+
only showing top 2 rows


In [17]:
heart_df = heart_df.fillna(0)

In [ ]:
assembler = VectorAssembler(
    inputCols=[
        "male", "age", "education", "currentSmoker", "cigsPerDay", "BPMeds", 
        "prevalentStroke", "prevalentHyp", "diabetes", "totChol", 
        "sysBP", "diaBP", "BMI", "heartRate", "glucose"
    ],
    outputCol="features"
)

heart_df = heart_df.withColumnRenamed("TenYearCHD", "label")

features = assembler.transform(heart_df).select("label", "features")# renaming target column to "label"

features.printSchema()  

root
 |-- label: integer (nullable = true)
 |-- features: vector (nullable = true)



In [22]:
features.show(10)

+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|[1.0,39.0,4.0,0.0...|
|    0|(15,[1,2,9,10,11,...|
|    0|[1.0,48.0,1.0,1.0...|
|    1|[0.0,61.0,3.0,1.0...|
|    0|[0.0,46.0,3.0,1.0...|
|    0|[0.0,43.0,2.0,0.0...|
|    1|(15,[1,2,9,10,11,...|
|    0|[0.0,45.0,2.0,1.0...|
|    0|[1.0,52.0,1.0,0.0...|
|    0|[1.0,43.0,1.0,1.0...|
+-----+--------------------+
only showing top 10 rows


# Data Splitting

In [24]:
train_df, test_df = features.randomSplit([0.8, 0.2], seed=50)

In [27]:
print("Original Dataset")
# heart_df.show()

# Print train dataset
print("train set")
train_df.show()

Original Dataset
train set
+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
|    0|(15,[0,1,2,9,10,1...|
+-----+--------------------+
only showing top 20 rows


# Create ML Model

In [28]:
lr = LogisticRegression(maxIter=10, regParam=0.01)

# Train ML Model

In [29]:
lr_model = lr.fit(train_df)

# Print coefficients
print("Coefficients: " + str(lr_model.coefficients))

# Display model summary
training_summary = lr_model.summary

Coefficients: [0.41580512117107044,0.062386283516254706,-0.03043254434534699,0.04040238478001188,0.02041062841298868,-0.07804561485635934,0.848980486010651,0.314576686899524,0.5756825161535588,9.473042763450876e-05,0.012642103891135612,0.0015514069992430236,-0.009562138788549931,-0.0021140920934678937,0.0033356178444707988]


### Prediction

In [30]:
# Use the trained model to make predictions on the test data
predictions = lr_model.transform(test_df)

# Show predictions
predictions.select("features", "prediction", "probability").show()

+--------------------+----------+--------------------+
|            features|prediction|         probability|
+--------------------+----------+--------------------+
|(15,[0,1,2,9,10,1...|       0.0|[0.91353043301868...|
|(15,[0,1,2,9,10,1...|       0.0|[0.88742101875731...|
|(15,[0,1,9,10,11,...|       0.0|[0.79436914346864...|
|(15,[0,1,9,10,11,...|       0.0|[0.73930987388464...|
|(15,[0,1,10,11,12...|       0.0|[0.69145056395169...|
|(15,[1,2,3,4,10,1...|       0.0|[0.95573080588886...|
|(15,[1,2,3,4,10,1...|       0.0|[0.90612184415217...|
|(15,[1,2,7,9,10,1...|       0.0|[0.89505372592777...|
|(15,[1,2,7,9,10,1...|       0.0|[0.68826461798094...|
|(15,[1,2,9,10,11,...|       0.0|[0.97937616557049...|
|(15,[1,2,9,10,11,...|       0.0|[0.96693480350247...|
|(15,[1,2,9,10,11,...|       0.0|[0.96493185560978...|
|(15,[1,2,9,10,11,...|       0.0|[0.96354588468188...|
|(15,[1,2,9,10,11,...|       0.0|[0.96543285679648...|
|(15,[1,2,9,10,11,...|       0.0|[0.96613848986306...|
|(15,[1,2,

# Test ML Model

In [31]:
evaluator = MulticlassClassificationEvaluator(labelCol="label",
                            predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, 
                  {evaluator.metricName: "accuracy"})
print(f"Accuracy: {accuracy}")
precision = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedPrecision"})
print(f"Precision: {precision}")
recall = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedRecall"})
print(f"Recall: {recall}")
f1 = evaluator.evaluate(predictions,
                {evaluator.metricName: "f1"})
print(f"F1 Score: {f1}")  

Accuracy: 0.8327228327228328
Precision: 0.802768866744022
Recall: 0.8327228327228327
F1 Score: 0.7713557304578122


In [32]:
spark.stop()